In [0]:
catalog = "workspace"
schema = "retail_lab"
base = f"/Volumes/{catalog}/{schema}/raw_files"

customers = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base}/customers.csv"))

products = (spark.read
    .option("header", True)
        .option("inferSchema", True)
    .csv(f"{base}/products.csv"))

orders = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base}/orders.csv"))

display(customers.limit(5))
display(products.limit(5))
display(orders.limit(5))

customer_id,customer_name,email,city,region,signup_date,customer_segment
C0001,Customer_001,customer1@example.com,Guwahati,East,2023-03-31,Small Business
C0002,Customer_002,customer2@example.com,Pune,West,2025-02-12,Small Business
C0003,Customer_003,customer3@example.com,Kochi,South,2024-10-16,Small Business
C0004,Customer_004,customer4@example.com,Surat,West,2024-03-14,Corporate
C0005,Customer_005,customer5@example.com,Nagpur,Central,2024-03-09,Corporate


product_id,product_name,category,unit_cost,list_price
P001,Wireless Mouse,Electronics,8047.36,8559.07
P002,Keyboard,Electronics,6817.61,13805.23
P003,Monitor,Electronics,7645.84,10031.39
P004,Headphones,Electronics,6745.75,5575.51
P005,Webcam,Electronics,447.35,7945.41


order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,order_status,shipping_date,payment_method
O00001,2025-04-18,C0032,P017,7,6951.23,0.0,Cancelled,null,UPI
O00002,2026-03-17,C0089,P006,1,7442.24,0.0,Delivered,2026-03-21,Cash on Delivery
O00003,2026-07-23,C0087,P007,6,4325.82,0.1,Pending,null,UPI
O00004,2024-01-21,C0092,P011,6,7404.01,0.1,Delivered,2024-01-25,UPI
O00005,2024-04-08,C0078,P014,1,8394.51,0.15,Pending,null,Cash on Delivery


In [0]:
#convert csv to parquet
parquet_base = f"/Volumes/{catalog}/{schema}/parquet"

customers.write.mode("overwrite").parquet(f"{parquet_base}/customers")
products.write.mode("overwrite").parquet(f"{parquet_base}/products")
orders.write.mode("overwrite").parquet(f"{parquet_base}/orders")

# Read the Parquet data back
orders_parquet = spark.read.parquet(f"{parquet_base}/orders")
orders_parquet.printSchema()
display(orders_parquet.limit(10))

customers_parquet = spark.read.parquet(f"{parquet_base}/customers")
products_parquet = spark.read.parquet(f"{parquet_base}/products")

customers_parquet.printSchema()



root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- order_status: string (nullable = true)
 |-- shipping_date: date (nullable = true)
 |-- payment_method: string (nullable = true)



order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,order_status,shipping_date,payment_method
O00001,2025-04-18,C0032,P017,7,6951.23,0.0,Cancelled,null,UPI
O00002,2026-03-17,C0089,P006,1,7442.24,0.0,Delivered,2026-03-21,Cash on Delivery
O00003,2026-07-23,C0087,P007,6,4325.82,0.1,Pending,null,UPI
O00004,2024-01-21,C0092,P011,6,7404.01,0.1,Delivered,2024-01-25,UPI
O00005,2024-04-08,C0078,P014,1,8394.51,0.15,Pending,null,Cash on Delivery
O00006,2025-03-06,C0023,P007,5,4325.82,0.0,Delivered,2025-03-11,UPI
O00007,2024-04-17,C0036,P020,7,17319.87,0.1,Delivered,2024-04-21,Net Banking
O00008,2024-12-25,C0019,P021,3,12639.77,0.05,Shipped,2024-12-29,Debit Card
O00009,2026-03-21,C0081,P011,4,7404.01,0.2,Delivered,2026-03-23,Net Banking
O00010,2025-08-14,C0115,P007,4,4325.82,0.1,Delivered,null,Net Banking


root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)



In [0]:
#Level 1 – Basics
#Display the first 10 orders
orders.show(10, truncate=False)

+--------+----------+-----------+----------+--------+----------+------------+------------+-------------+----------------+
|order_id|order_date|customer_id|product_id|quantity|unit_price|discount_pct|order_status|shipping_date|payment_method  |
+--------+----------+-----------+----------+--------+----------+------------+------------+-------------+----------------+
|O00001  |2025-04-18|C0032      |P017      |7       |6951.23   |0.0         |Cancelled   |NULL         |UPI             |
|O00002  |2026-03-17|C0089      |P006      |1       |7442.24   |0.0         |Delivered   |2026-03-21   |Cash on Delivery|
|O00003  |2026-07-23|C0087      |P007      |6       |4325.82   |0.1         |Pending     |NULL         |UPI             |
|O00004  |2024-01-21|C0092      |P011      |6       |7404.01   |0.1         |Delivered   |2024-01-25   |UPI             |
|O00005  |2024-04-08|C0078      |P014      |1       |8394.51   |0.15        |Pending     |NULL         |Cash on Delivery|
|O00006  |2025-03-06|C00

In [0]:
#Show the schema and row count of each dataset
# Create a dictionary of datasets
datasets = {
    "Customers": customers,
    "Products": products,
    "Orders": orders
}

# Display schema and row count
for name, df in datasets.items():

    print(f"\n========== {name} ==========")

    # Display schema
    print("Schema:")
    df.printSchema()

    # Display row count
    print("Total Rows:", df.count())


========== Customers ==========
Schema:
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)

Total Rows: 120

========== Products ==========
Schema:
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_cost: double (nullable = true)
 |-- list_price: double (nullable = true)

Total Rows: 21

========== Orders ==========
Schema:
root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- order_status: string (nullable = true)


In [0]:
#Find all orders with status Delivered
delivered_orders = orders.filter(orders.order_status == "Delivered")

display(delivered_orders)

order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,order_status,shipping_date,payment_method
O00002,2026-03-17,C0089,P006,1,7442.24,0.0,Delivered,2026-03-21,Cash on Delivery
O00004,2024-01-21,C0092,P011,6,7404.01,0.1,Delivered,2024-01-25,UPI
O00006,2025-03-06,C0023,P007,5,4325.82,0.0,Delivered,2025-03-11,UPI
O00007,2024-04-17,C0036,P020,7,17319.87,0.1,Delivered,2024-04-21,Net Banking
O00009,2026-03-21,C0081,P011,4,7404.01,0.2,Delivered,2026-03-23,Net Banking
O00010,2025-08-14,C0115,P007,4,4325.82,0.1,Delivered,null,Net Banking
O00011,2025-03-30,C0002,P019,4,1714.55,0.0,Delivered,2025-04-02,Credit Card
O00012,2025-07-27,C0015,P006,2,7442.24,0.1,Delivered,2025-08-02,UPI
O00013,2026-01-22,C0074,P020,7,17319.87,0.0,Delivered,2026-01-24,Net Banking
O00014,2024-02-08,C0038,P018,4,11400.54,0.15,Delivered,2024-02-15,Debit Card


In [0]:
#Show orders placed in 2025.
from pyspark.sql.functions import col

orders_2025 = orders.filter(
    (col("order_date") >= "2025-01-01") &
    (col("order_date") < "2026-01-01")
)

display(orders_2025)

order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,order_status,shipping_date,payment_method
O00001,2025-04-18,C0032,P017,7,6951.23,0.0,Cancelled,null,UPI
O00006,2025-03-06,C0023,P007,5,4325.82,0.0,Delivered,2025-03-11,UPI
O00010,2025-08-14,C0115,P007,4,4325.82,0.1,Delivered,null,Net Banking
O00011,2025-03-30,C0002,P019,4,1714.55,0.0,Delivered,2025-04-02,Credit Card
O00012,2025-07-27,C0015,P006,2,7442.24,0.1,Delivered,2025-08-02,UPI
O00033,2025-01-05,C0018,P005,6,7945.41,0.0,Shipped,2025-01-08,Debit Card
O00034,2025-09-11,C0020,P004,2,5575.51,0.1,Shipped,2025-09-13,Net Banking
O00037,2025-06-06,C0065,P001,4,8559.07,0.0,Delivered,2025-06-13,Net Banking
O00039,2025-01-21,C0012,P018,1,11400.54,0.0,Delivered,2025-01-24,UPI
O00041,2025-10-11,C0078,P013,6,1757.35,0.15,Delivered,2025-10-12,Cash on Delivery


In [0]:
#List all distinct order statuses.
distinct_statuses = orders.select("order_status").distinct()

display(distinct_statuses)

order_status
Pending
Shipped
Returned
Delivered
Cancelled


In [0]:
#Find orders with a missing shipping_date
from pyspark.sql.functions import col

missing_orders = orders_parquet.filter(
    col("shipping_date").isNull()
)

display(missing_orders)

order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,order_status,shipping_date,payment_method
O00001,2025-04-18,C0032,P017,7,6951.23,0.0,Cancelled,null,UPI
O00003,2026-07-23,C0087,P007,6,4325.82,0.1,Pending,null,UPI
O00005,2024-04-08,C0078,P014,1,8394.51,0.15,Pending,null,Cash on Delivery
O00010,2025-08-14,C0115,P007,4,4325.82,0.1,Delivered,null,Net Banking
O00019,2026-02-11,C0114,P006,1,7442.24,0.0,Pending,null,Net Banking
O00028,2026-02-03,C0097,P010,4,14142.58,0.1,Delivered,null,UPI
O00035,2026-01-03,C0007,P005,2,7945.41,0.15,Cancelled,null,Debit Card
O00040,2026-08-15,C0097,P019,2,1714.55,0.2,Pending,null,Cash on Delivery
O00049,2025-06-25,C0113,P016,1,12579.32,0.2,Pending,null,Debit Card
O00055,2026-03-12,C0089,P019,6,1714.55,0.15,Cancelled,null,Debit Card


In [0]:
#Level 2 – Aggregations
#1. Calculate the total number of orders by status
from pyspark.sql.functions import count

orders_by_status = (
    orders_parquet
    .groupBy("order_status")
    .agg(count("*").alias("total_orders"))
    .orderBy("order_status")
)

display(orders_by_status)

order_status,total_orders
Cancelled,114
Delivered,938
Pending,138
Returned,82
Shipped,228


In [0]:
#Calculate total quantity sold by product.
from pyspark.sql.functions import sum, col

quantity_by_product = (
    orders_parquet
    .groupBy("product_id")
    .agg(
        sum("quantity").alias("total_quantity_sold")
    )
    .orderBy(col("total_quantity_sold").desc())
)

display(quantity_by_product)

product_id,total_quantity_sold
P003,359
P011,358
P021,348
P015,331
P010,299
P001,299
P012,292
P006,290
P014,289
P008,287


In [0]:
#Calculate revenue by payment method. Revenue = quantity × unit_price × (1 − discount_pct). Treat a missing discount as 0.
from pyspark.sql.functions import col, sum, coalesce, lit

revenue_by_payment = (
    orders_parquet
    .withColumn(
        "discount_pct",
        coalesce(col("discount_pct"), lit(0))
    )
    .withColumn(
        "revenue",
        col("quantity") * col("unit_price") * (1 - col("discount_pct"))
    )
    .groupBy("payment_method")
    .agg(
        sum("revenue").alias("total_revenue")
    )
    .orderBy(col("total_revenue").desc())
)

display(revenue_by_payment)

payment_method,total_revenue
Cash on Delivery,1.049678575E7
Net Banking,9721743.590499995
Credit Card,9285677.679999996
Debit Card,8893619.037999993
UPI,8409202.554000001


In [0]:
from pyspark.sql.functions import (
    col, sum, coalesce, lit,
    year, month
)

monthly_revenue = (
    orders_parquet
    .filter(year(col("order_date")) == 2025)
    .withColumn(
        "discount_pct",
        coalesce(col("discount_pct"), lit(0))
    )
    .withColumn(
        "revenue",
        col("quantity") * col("unit_price") *
        (1 - col("discount_pct"))
    )
    .groupBy(month(col("order_date")).alias("month"))
    .agg(
        sum("revenue").alias("total_revenue")
    )
    .orderBy("month")
)

display(monthly_revenue)

month,total_revenue
1,1326127.7465000001
2,1012008.9110000001
3,1840762.3345
4,1373788.2655000004
5,1371503.4525000001
6,1625915.3590000004
7,1357006.7854999995
8,1427293.0535000002
9,1429870.8489999997
10,1349646.6030000004


In [0]:
#Find the top 5 customers by total order value.

from pyspark.sql.functions import col, sum, coalesce, lit

top_5_customers = (
    orders_parquet
    .withColumn(
        "discount_pct",
        coalesce(col("discount_pct"), lit(0))
    )
    .withColumn(
        "order_value",
        col("quantity") * col("unit_price") *
        (1 - col("discount_pct"))
    )
    .groupBy("customer_id")
    .agg(
        sum("order_value").alias("total_order_value")
    )
    .orderBy(col("total_order_value").desc())
    .limit(5)
)

display(top_5_customers)

customer_id,total_order_value
C0074,874382.8165
C0019,721784.821
C0029,677281.0255
C0018,624319.5889999999
C0013,617114.2995000001


In [0]:
#Calculate average order value by customer segment

from pyspark.sql.functions import (
    col, sum, avg, coalesce, lit
)

# Step 1: Calculate order value
orders_with_revenue = (
    orders_parquet
    .withColumn(
        "discount_pct",
        coalesce(col("discount_pct"), lit(0))
    )
    .withColumn(
        "order_value",
        col("quantity") * col("unit_price") *
        (1 - col("discount_pct"))
    )
)

# Step 2: Join orders with customers
orders_with_segment = (
    orders_with_revenue
    .join(customers_parquet, on="customer_id", how="inner")
)

# Step 3: Calculate average order value by customer segment
avg_order_by_segment = (
    orders_with_segment
    .groupBy("customer_segment")
    .agg(
        avg("order_value").alias("average_order_value")
    )
    .orderBy(col("average_order_value").desc())
)

display(avg_order_by_segment)

customer_segment,average_order_value
Corporate,32854.83244913421
Consumer,30727.976998087965
Small Business,30208.473885436866


In [0]:
#Level 3 – Joins and data quality
#Join orders to customers and show order ID, customer name, city and region.
from pyspark.sql.functions import col

orders_customers = (
    orders_parquet
    .join(
        customers_parquet,
        on="customer_id",
        how="inner"
    )
    .select(
        "order_id",
        "customer_name",
        "city",
        "region"
    )
)

display(orders_customers)

order_id,customer_name,city,region
O00001,Customer_032,Delhi,North
O00002,Customer_089,Guwahati,East
O00003,Customer_087,Hyderabad,South
O00004,Customer_092,Delhi,North
O00005,Customer_078,Raipur,Central
O00006,Customer_023,Ahmedabad,West
O00007,Customer_036,Raipur,Central
O00008,Customer_019,Chennai,South
O00009,Customer_081,Mumbai,West
O00010,Customer_115,Surat,West


In [0]:
#Join orders to products and calculate revenue by product category.
from pyspark.sql.functions import col, sum, coalesce, lit

# Step 1: Join orders with products
orders_products = (
    orders_parquet
    .join(
        products_parquet,
        on="product_id",
        how="inner"
    )
)

# Step 2: Calculate revenue
orders_products = (
    orders_products
    .withColumn(
        "discount_pct",
        coalesce(col("discount_pct"), lit(0))
    )
    .withColumn(
        "revenue",
        col("quantity") * col("unit_price") *
        (1 - col("discount_pct"))
    )
)

# Step 3: Calculate revenue by product category
revenue_by_category = (
    orders_products
    .groupBy("category")
    .agg(
        sum("revenue").alias("total_revenue")
    )
    .orderBy(col("total_revenue").desc())
)

display(revenue_by_category)

category,total_revenue
Electronics,1.425284365400001E7
Accessories,1.1867239021499986E7
Furniture,1.1369207079499992E7
Stationery,9317738.857499992


In [0]:
#Find customers who have never placed an order.
customers_without_orders = (
    customers_parquet
    .join(
        orders_parquet,
        on="customer_id",
        how="left_anti"
    )
)

display(customers_without_orders)

customer_id,customer_name,email,city,region,signup_date,customer_segment


In [0]:
#Find customers who have never placed an order.
from pyspark.sql.functions import col

customers_without_orders = (
    customers_parquet
    .join(
        orders_parquet,
        on="customer_id",
        how="left"
    )
    .filter(col("order_id").isNull())
    .select(customers_parquet["*"])
)

display(customers_without_orders)

customer_id,customer_name,email,city,region,signup_date,customer_segment


In [0]:
#22.	Identify duplicate order IDs.
from pyspark.sql.functions import col, count

duplicate_orders = (
    orders_parquet
    .groupBy("order_id")
    .agg(
        count("*").alias("duplicate_count")
    )
    .filter(col("duplicate_count") > 1)
    .orderBy(col("duplicate_count").desc())
)

display(duplicate_orders)

order_id,duplicate_count


In [0]:
#23.	Replace missing discounts with 0 and verify there are no NULL discounts.
#step1
from pyspark.sql.functions import col, coalesce, lit

orders_cleaned = (
    orders_parquet
    .withColumn(
        "discount_pct",
        coalesce(col("discount_pct"), lit(0))
    )
)

display(orders_cleaned)

order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,order_status,shipping_date,payment_method
O00001,2025-04-18,C0032,P017,7,6951.23,0.0,Cancelled,null,UPI
O00002,2026-03-17,C0089,P006,1,7442.24,0.0,Delivered,2026-03-21,Cash on Delivery
O00003,2026-07-23,C0087,P007,6,4325.82,0.1,Pending,null,UPI
O00004,2024-01-21,C0092,P011,6,7404.01,0.1,Delivered,2024-01-25,UPI
O00005,2024-04-08,C0078,P014,1,8394.51,0.15,Pending,null,Cash on Delivery
O00006,2025-03-06,C0023,P007,5,4325.82,0.0,Delivered,2025-03-11,UPI
O00007,2024-04-17,C0036,P020,7,17319.87,0.1,Delivered,2024-04-21,Net Banking
O00008,2024-12-25,C0019,P021,3,12639.77,0.05,Shipped,2024-12-29,Debit Card
O00009,2026-03-21,C0081,P011,4,7404.01,0.2,Delivered,2026-03-23,Net Banking
O00010,2025-08-14,C0115,P007,4,4325.82,0.1,Delivered,null,Net Banking


In [0]:
#Step 2: Verify there are no NULL discounts
null_discount_count = (
    orders_cleaned
    .filter(col("discount_pct").isNull())
    .count()
)

print("NULL discounts:", null_discount_count)

NULL discounts: 0


In [0]:
#Calculate the percentage of orders in each status.
from pyspark.sql.functions import col, count, round

# Step 1: Calculate total number of orders
total_orders = orders_parquet.count()

# Step 2: Calculate orders by status and percentage
status_percentage = (
    orders_parquet
    .groupBy("order_status")
    .agg(
        count("*").alias("total_orders")
    )
    .withColumn(
        "percentage",
        round(
            col("total_orders") / total_orders * 100,
            2
        )
    )
    .orderBy(col("percentage").desc())
)

display(status_percentage)

order_status,total_orders,percentage
Delivered,938,62.53
Shipped,228,15.2
Pending,138,9.2
Cancelled,114,7.6
Returned,82,5.47
